# TF-IDF + Logistic Regression Baseline

This notebook implements a baseline model for sarcasm detection using TF-IDF and logistic regression.

### 1. Import required libraries

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer # convert text to numerical features so logistic regression can learn from it
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report # performance metrics

### 2. Load datasets

In [15]:
train_df = pd.read_csv('sarcasm_train.csv')
test_df = pd.read_csv('sarcasm_test.csv')

print(f"Data loaded successfully. Training set size: {len(train_df)}, Test set size: {len(test_df)}")

Data loaded successfully. Training set size: 807972, Test set size: 201992


### 3. Extract input text (`X`) and labels (`y`)

In [16]:
X_train_text = train_df['combined_text'].fillna('')
y_train = train_df['label']

X_test_text = test_df['combined_text'].fillna('')
y_test = test_df['label']

# show the proportion of each label (0 and 1) in the training set.
print(y_train.value_counts(normalize=True).sort_index())

label
0    0.499397
1    0.500603
Name: proportion, dtype: float64


### 4. Vectorize text and train the logistic regression model.

In [23]:
vectorizer = TfidfVectorizer(
    # keep only the 10000 most common words/phrases as features
    max_features=10000,

    # use both single words and 2-word phrases as features ("not good" more informative than "not" and "good")
    ngram_range=(1, 2),
)

# Convert training and test text into TF-IDF features
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# Create the logistic regression classifier
clf = LogisticRegression(
    max_iter=2000,

    # liblinear is commonly used for small datasets and binary classification
    solver='liblinear',

    # use a fixed random seed so results are reproducible
    random_state=10
)

# train the model
clf.fit(X_train, y_train)

# predict on the test set
y_pred = clf.predict(X_test)

print('Model training completed')

Model training complete.


### 5. Evaluate Model Performance

In [26]:
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print(f'Accuracy: {acc:.4f}')
print(f'F1 Score (Macro): {f1:.4f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.6709
F1 Score (Macro): 0.6704

Classification report:
              precision    recall  f1-score   support

           0       0.66      0.71      0.68    101412
           1       0.68      0.63      0.66    100580

    accuracy                           0.67    201992
   macro avg       0.67      0.67      0.67    201992
weighted avg       0.67      0.67      0.67    201992



### 6. Diagnostic Analysis: Top weighted words/features (most associated with sarcastic vs non-sarcastic predictions)

In [ ]:
# get all the words/phrases the TF-IDF vectorizer learned
feature_names = vectorizer.get_feature_names_out()

# get the logistic regression weight for each feature
# positive weight pushes prediction toward label 1
# negative weight pushes prediction toward label 0
coefs = clf.coef_[0]

# find the 20 features with the largest positive weights
top_pos_idx = np.argsort(coefs)[-20:][::-1]

# find the 20 features with the most negative weights
top_neg_idx = np.argsort(coefs)[:20]

# Show the features that make the model more likely to predict label 1
print('Top 20 words/phrases that push the prediction toward label = 1:')
for i in top_pos_idx:
    print(feature_names[i], ':', round(coefs[i], 4))

# Show the features that make the model more likely to predict label 0
print('\nTop 20 words/phrases that push the prediction toward label = 0:')
for i in top_neg_idx:
    print(feature_names[i], ':', round(coefs[i], 4))

Top 20 words/phrases that push the prediction toward label = 1:
yes because : 9.5234
yeah because : 8.008
obviously : 7.9661
forgot the : 7.4529
but but : 7.0973
dropped this : 6.9283
you dropped : 6.8537
duh : 6.6656
clearly : 6.6168
how dare : 6.5993
totally : 5.7748
forgot your : 5.7604
shitlord : 5.3859
yeah : 5.3275
right because : 5.3191
forgot this : 5.2732
gee : 5.1885
but thought : 5.0961
because : 4.9936
good thing : 4.7214

Top 20 words/phrases that push the prediction toward label = 0:
but yeah : -3.4962
iirc : -3.4959
not sure : -3.4683
forgot about : -3.3498
never said : -3.2224
so yeah : -2.8125
true but : -2.644
sure but : -2.5853
psn : -2.5294
right now : -2.4714
as well : -2.4012
right but : -2.3685
the original : -2.3265
for sure : -2.3223
pretty sure : -2.2874
make sure : -2.2735
that but : -2.2727
them but : -2.2523
there but : -2.2405
up but : -2.1982
